# Notebook 03 · Serve Qwen3.5-2B from Colab for the assistant (temporary endpoint)

Use this for a **live demo** of the app's *Server LLM* mode. It runs Qwen3.5-2B (Apache-2.0, 2 B parameters)
with Ollama on the Colab GPU and exposes an **OpenAI-compatible** endpoint through a Cloudflare quick tunnel.

* The URL changes every time and stops when the Colab session ends — good for a demo, not for production.
* For an always-on endpoint use the Hugging Face Space in `llm-server/` (llama.cpp, API key protected),
  or skip servers entirely with the app's **In-browser (WebGPU)** mode.
* The quick tunnel has **no authentication**; anyone with the URL can use the model while it runs.
  Stop the cell (or the runtime) when you are done.

In [ ]:
MODEL = "hf.co/unsloth/Qwen3.5-2B-GGUF:Q4_K_M"   # GGUF build of Qwen/Qwen3.5-2B

In [ ]:
import os, re, subprocess, time, json, urllib.request
subprocess.run("apt-get -qq install -y zstd > /dev/null 2>&1", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
env = {**os.environ, "OLLAMA_HOST": "0.0.0.0:11434", "OLLAMA_KEEP_ALIVE": "24h"}
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
time.sleep(8)
subprocess.run(["ollama", "pull", MODEL], check=True)

In [ ]:
def chat(base, question, key="ollama"):
    body = json.dumps({"model": MODEL, "temperature": 0.2, "max_tokens": 120,
                       "messages": [{"role": "user", "content": question}]}).encode()
    req = urllib.request.Request(base + "/chat/completions", data=body,
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}"})
    return json.loads(urllib.request.urlopen(req, timeout=120).read())["choices"][0]["message"]["content"]

print(chat("http://localhost:11434/v1", "In one sentence, what is a building's baseload?"))

In [ ]:
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
               "-O cloudflared && chmod +x cloudflared", shell=True, check=True)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434",
                           "--http-host-header", "localhost:11434", "--no-autoupdate"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0); break
print("Public URL:", url)
print("\nSet these in Vercel → Project → Settings → Environment Variables, then redeploy:")
print(f"  LLM_BASE_URL = {url}/v1")
print(f"  LLM_MODEL    = {MODEL}")
print( "  LLM_API_KEY  = ollama")
print("\nOr test locally:  LLM_BASE_URL=... npm run dev")

In [ ]:
print(chat(url + "/v1", "Reply with the single word: ready"))